# Generic RAG Ingestion Pipeline


In [ ]:
#uv add section
!uv add ipykernel nltk sentence-transformers ipywidgets chromadb --active


Resolved 95 packages in 415ms
 Downloaded widgetsnbextension
Prepared 3 packages in 581ms
Installed 3 packages in 62ms
 + ipywidgets==8.1.8
 + jupyterlab-widgets==3.0.16
 + widgetsnbextension==4.0.15


Setup Python environment. I hope you have been using uv by now. If not (why??), then just pip install it.

In [ ]:
# Setup virtual environment using uv
!uv venv .venv
!uv sync --active

First we need to extract text from raw data sources. In this case, we use PyMuPDF.

In [2]:

from pathlib import Path
from typing import Dict
import fitz  # PyMuPDF

# Extract Text from PDF
def extract_text_from_pdf(pdf_path: Path) -> Dict[int, str]:
    doc = fitz.open(str(pdf_path))
    pages = {}

    for page_num in range(len(doc)):
        page = doc[page_num]
        # sort=True reads text in visual top-to-bottom, left-to-right order,
        # fixing clipped/interleaved characters common in PDFs with columns or TOC pages
        text = page.get_text("text", sort=True).strip()
        if text:
            pages[page_num + 1] = text

    doc.close()
    print(f"Extracted text from {len(pages)} pages in {pdf_path.name}")
    return pages

# Put the path to your PDF file here
extracted_text = extract_text_from_pdf(pdf_path=Path("Docs/Student-Handbook-2024_v2.pdf"))


Extracted text from 216 pages in Student-Handbook-2024_v2.pdf


Then split the text into chunks

In [31]:
import hashlib
import nltk
from typing import List, Dict, Any

nltk.download("punkt_tab", quiet=True)

# Configuration — adjust these as needed
CHUNK_SIZE = 1000       # max characters per chunk
CHUNK_OVERLAP = 200     # overlap between consecutive chunks (in characters)
MIN_CHUNK_SIZE = 50     # discard chunks smaller than this


def chunk_text(
    text: str,
    source: str = "",
    source_path: str = "",
    page: int = 0,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
) -> List[Dict[str, Any]]:
    """
    Sentence-aware text chunking. Splits text into sentences first,
    then groups sentences into chunks that respect chunk_size.
    Overlap is achieved by re-including sentences from the end of the
    previous chunk at the start of the next one.

    Args:
        text:         Text to chunk.
        source:       Source filename (e.g. "manual.pdf").
        source_path:  Full path to source file.
        page:         Page number within the source document.
        chunk_size:   Target maximum characters per chunk.
        chunk_overlap: Approximate character overlap between chunks.

    Returns:
        List of chunk dicts with text and metadata.
    """
    if not text or len(text) < MIN_CHUNK_SIZE:
        return []

    # Split into sentences; fall back to paragraph splits if nltk fails
    try:
        sentences = nltk.sent_tokenize(text)
    except Exception:
        sentences = [s.strip() for s in text.split("\n\n") if s.strip()]

    chunks: List[Dict[str, Any]] = []
    current_sentences: List[str] = []
    current_len = 0
    chunk_idx = 0
    char_cursor = 0

    for sentence in sentences:
        sentence_len = len(sentence)

        # If adding this sentence would exceed chunk_size, flush current buffer
        if current_sentences and current_len + sentence_len > chunk_size:
            chunk_text_str = " ".join(current_sentences).strip()
            if len(chunk_text_str) >= MIN_CHUNK_SIZE:
                char_start = char_cursor - current_len
                chunks.append({
                    "text": chunk_text_str,
                    "source": source,
                    "source_path": source_path,
                    "page": page,
                    "chunk_index": chunk_idx,
                    "char_start": max(0, char_start),
                    "char_end": char_cursor,
                    "hash": hashlib.md5(chunk_text_str.encode()).hexdigest(),
                })
                chunk_idx += 1

            # Build overlap: roll back sentences until we're within overlap budget
            overlap_sentences: List[str] = []
            overlap_len = 0
            for s in reversed(current_sentences):
                if overlap_len + len(s) > chunk_overlap:
                    break
                overlap_sentences.insert(0, s)
                overlap_len += len(s) + 1  # +1 for the space

            current_sentences = overlap_sentences
            current_len = overlap_len

        current_sentences.append(sentence)
        current_len += sentence_len + 1  # +1 for joining space
        char_cursor += sentence_len + 1

    # Flush any remaining sentences
    if current_sentences:
        chunk_text_str = " ".join(current_sentences).strip()
        if len(chunk_text_str) >= MIN_CHUNK_SIZE:
            chunks.append({
                "text": chunk_text_str,
                "source": source,
                "source_path": source_path,
                "page": page,
                "chunk_index": chunk_idx,
                "char_start": max(0, char_cursor - current_len),
                "char_end": char_cursor,
                "hash": hashlib.md5(chunk_text_str.encode()).hexdigest(),
            })

    return chunks

source_filename = "Student-Handbook-2024_v2.pdf"
source_path = f"Docs/{source_filename}"

# chunk_text() works on one page at a time, so iterate over all extracted pages
all_chunks = []
for page_num, page_text in extracted_text.items():
    chunks = chunk_text(
        text=page_text,
        source=source_filename,
        source_path=source_path,
        page=page_num,
    )
    all_chunks.extend(chunks)

print(f"Total chunks: {len(all_chunks)}")
print(f"\n--- Example chunk ---")

for i, chunk in enumerate(all_chunks):
    print(f"[{i}] Page {chunk['page']} | Chunk {chunk['chunk_index']} | {len(chunk['text'])} chars")
    print(f"     {chunk['text'][:500].replace(chr(10), ' ')}...")
    print()



Total chunks: 497

--- Example chunk ---
[0] Page 3 | Chunk 0 | 1714 chars
     ASIA PACIFIC COLLEGE      STUDENT HANDBOOK         Contents  FOREWORD                                                8  HISTORY                                                  9  MISSION, VISION, AND VALUES                             10  GRADUATE ATTRIBUTES                                     11  GUIDING PRINCIPLES                                      12  QUALITY POLICY                                         13  ARTIFICIAL INTELLIGENCE (AI) MANIFESTO                  14  GLOSSARY             ...

[1] Page 4 | Chunk 0 | 402 chars
     STUDENT SERVICES  I. ADMISSIONS OFFICE PLAYBOOK                          24       Scholarship Grants                                      24     Financial Assistance                                     26    Terms and Conditions                                    27     Engineering Scholarship Program                          29    Renewal of Grants/Financial Assistance     

Or as an alternative, use semantic chunking (copilot generated example)

In [4]:
import hashlib
import nltk
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any

nltk.download("punkt_tab", quiet=True)

# --- Config ---
EMBED_MODEL = "all-MiniLM-L6-v2"   # fast, good quality — swap for a larger model if needed
BREAKPOINT_PERCENTILE = 85          # sentences whose similarity drop exceeds this percentile are cut points
MIN_CHUNK_SENTENCES = 2             # don't emit chunks smaller than this many sentences
MAX_CHUNK_SENTENCES = 20            # force a cut after this many sentences even without a breakpoint

model = SentenceTransformer(EMBED_MODEL)


def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))


def semantic_chunk_document(
    pages: Dict[int, str],
    source: str = "",
    source_path: str = "",
    breakpoint_percentile: int = BREAKPOINT_PERCENTILE,
    min_sentences: int = MIN_CHUNK_SENTENCES,
    max_sentences: int = MAX_CHUNK_SENTENCES,
) -> List[Dict[str, Any]]:
    """
    Semantically chunks an entire document by detecting topic-shift boundaries.

    Steps:
      1. Join all pages and split into sentences.
      2. Embed every sentence with a small transformer model.
      3. Compute cosine similarity between each adjacent pair of sentences.
      4. Treat similarity *drops* above a threshold as chunk boundaries.
      5. Group sentences between boundaries into chunks.

    Args:
        pages:                 {page_num: text} dict from extract_text_from_pdf.
        source:                Source filename.
        source_path:           Full path to source file.
        breakpoint_percentile: Similarity drops above this percentile become cuts (0-100).
                               Lower = more chunks; higher = fewer, larger chunks.
        min_sentences:         Minimum sentences before a cut is allowed.
        max_sentences:         Force a cut after this many sentences regardless.

    Returns:
        List of chunk dicts with text and metadata.
    """
    full_text = "\n\n".join(text for _, text in sorted(pages.items()))

    try:
        sentences = nltk.sent_tokenize(full_text)
    except Exception:
        sentences = [s.strip() for s in full_text.split("\n\n") if s.strip()]

    sentences = [s for s in sentences if s.strip()]
    if len(sentences) < 2:
        return [{"text": full_text, "source": source, "source_path": source_path,
                 "chunk_index": 0, "hash": hashlib.md5(full_text.encode()).hexdigest()}]

    # Embed all sentences in one batched call
    print(f"Embedding {len(sentences)} sentences...")
    embeddings = model.encode(sentences, batch_size=64, show_progress_bar=True)

    # Similarity between each adjacent pair
    similarities = [
        cosine_similarity(embeddings[i], embeddings[i + 1])
        for i in range(len(embeddings) - 1)
    ]

    # A drop in similarity = topic shift; invert so high value = big drop
    drops = [1.0 - s for s in similarities]
    threshold = np.percentile(drops, breakpoint_percentile)

    # Find cut points (index i means "cut AFTER sentence i")
    cut_after: set = set()
    for i, drop in enumerate(drops):
        if drop >= threshold and i + 1 >= min_sentences:
            cut_after.add(i)

    # Also force cuts on max_sentences
    for i in range(max_sentences - 1, len(sentences) - 1, max_sentences):
        cut_after.add(i)

    # Build chunks
    chunks = []
    chunk_idx = 0
    start = 0
    for i in range(len(sentences)):
        if i in cut_after or i == len(sentences) - 1:
            group = sentences[start : i + 1]
            chunk_str = " ".join(group).strip()
            if chunk_str:
                chunks.append({
                    "text": chunk_str,
                    "source": source,
                    "source_path": source_path,
                    "chunk_index": chunk_idx,
                    "sentence_count": len(group),
                    "hash": hashlib.md5(chunk_str.encode()).hexdigest(),
                })
                chunk_idx += 1
            start = i + 1

    return chunks


# --- Usage ---
source_filename = "Student-Handbook-2024_v2.pdf"
semantic_chunks = semantic_chunk_document(
    pages=extracted_text,
    source=source_filename,
    source_path=f"Docs/{source_filename}",
)

print(f"\nTotal semantic chunks: {len(semantic_chunks)}")
for i, chunk in enumerate(semantic_chunks):
    print(f"[{i}] {chunk['sentence_count']} sentences | {len(chunk['text'])} chars")
    print(f"     {chunk['text'][:15000].replace(chr(10), ' ')}...")
    print()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding 2702 sentences...


Batches:   0%|          | 0/43 [00:00<?, ?it/s]


Total semantic chunks: 514
[0] 20 sentences | 9672 chars
     1  2  ASIA PACIFIC COLLEGE      STUDENT HANDBOOK         Contents  FOREWORD                                                8  HISTORY                                                  9  MISSION, VISION, AND VALUES                             10  GRADUATE ATTRIBUTES                                     11  GUIDING PRINCIPLES                                      12  QUALITY POLICY                                         13  ARTIFICIAL INTELLIGENCE (AI) MANIFESTO                  14  GLOSSARY                                              15  APC MARCH                                              16  GENERAL INFORMATION                                   17  Academic Calendar                                         17 Student Identification Cards                                  17 Shuttle Service                                            17 Shuttle Service Boarding Pass                                18 School Visitors          

## Step 3: Embed chunks and store in a vector database

Each chunk is embedded into a vector using the same model used for semantic chunking. The vectors + metadata are stored in ChromaDB (local, no server needed). At query time, the query is embedded with the same model and the nearest vectors are retrieved.


In [ ]:
import chromadb

# Use the same model that's already loaded from the semantic chunking cell.
# If you used sentence-aware chunking instead, initialise it here:
# from sentence_transformers import SentenceTransformer
# model = SentenceTransformer("all-MiniLM-L6-v2")

# --- ChromaDB setup (local, persisted to disk) ---
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Create (or load) a collection — safe to re-run, won't duplicate
collection = chroma_client.get_or_create_collection(
    name="rag_documents",
    metadata={"hnsw:space": "cosine"},   # use cosine similarity for retrieval
)

# --- Embed and ingest ---
# Use whichever chunk list you prefer: all_chunks (sentence-aware) or semantic_chunks
chunks_to_ingest = semantic_chunks   # swap to all_chunks if preferred

texts      = [c["text"]         for c in chunks_to_ingest]
ids        = [c["hash"]         for c in chunks_to_ingest]   # MD5 hash as stable unique ID
metadatas  = [
    {k: v for k, v in c.items() if k != "text"}              # store everything except the raw text
    for c in chunks_to_ingest
]

print(f"Embedding and ingesting {len(texts)} chunks...")
embeddings = model.encode(texts, batch_size=64, show_progress_bar=True).tolist()

collection.upsert(                    # upsert = insert or update — safe to re-run
    ids=ids,
    documents=texts,
    embeddings=embeddings,
    metadatas=metadatas,
)

print(f"Done. Collection now contains {collection.count()} documents.")


## Step 4: Retrieve — semantic search over stored chunks

Embed a natural language query with the same model, then ask ChromaDB for the top-k most similar chunks. These retrieved chunks become the **context** you pass to an LLM to generate a grounded answer.


In [ ]:
QUERY = "What are the scholarship requirements?"   # change this to anything
TOP_K = 5                                          # number of chunks to retrieve

query_embedding = model.encode(QUERY).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=TOP_K,
    include=["documents", "metadatas", "distances"],
)

print(f"Query: {QUERY!r}\n")
for rank, (doc, meta, dist) in enumerate(zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
)):
    similarity = 1 - dist   # ChromaDB returns cosine distance; convert to similarity
    print(f"[{rank + 1}] similarity={similarity:.3f} | chunk_index={meta.get('chunk_index')} | {len(doc)} chars")
    print(f"     {doc[:300].replace(chr(10), ' ')}...")
    print()

# The retrieved `docs` list is the context you pass to your LLM:
context = "\n\n---\n\n".join(results["documents"][0])
print("=== Context ready for LLM ===")
print(context[:500], "...")
